In [1]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
import torch

torch_dtype = torch.bfloat16
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"

initial_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch_dtype,
    attn_implementation="flash_attention_2",
    device_map={"": 0},
)

from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper

cell = MemoryCell(initial_model, num_mem_tokens=16)
model = RecurrentWrapper(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
# tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
# temp = tokenizer("test")
keys = ["input_ids", "attention_mask", "labels"]
test_input_data = {}
for key in keys:
    test_input_data[key] = torch.ones(
        (1, 2048),
        device=initial_model.device,
        dtype=torch.long,
    )
test_input_data

{'input_ids': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'labels': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}

In [4]:
model.memory_cell.model.dtype

torch.bfloat16

In [5]:
model.memory_cell.memory.dtype

torch.bfloat16

In [6]:
with torch.no_grad():
    result = model(**test_input_data)

result

CausalLMOutputWithCrossAttentions(loss=tensor(3.5000, device='cuda:0', dtype=torch.bfloat16), logits=tensor([[[ 5.5312,  6.3438,  4.9688,  ..., -1.5703, -1.5703, -1.5703],
         [ 6.2188,  9.5625,  5.4375,  ..., -1.9531, -1.9531, -1.9531],
         [ 7.2500, 10.3750,  6.2812,  ..., -1.8750, -1.8750, -1.8750],
         ...,
         [ 8.3125, 13.5000,  8.5625,  ..., -1.1875, -1.1875, -1.1875],
         [ 5.8750, 10.8125,  5.0000,  ..., -1.5859, -1.5859, -1.5859],
         [ 4.7500,  8.9375,  3.9375,  ..., -2.4219, -2.4219, -2.4219]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None, cross_attentions=None)

In [7]:
result.logits

tensor([[[ 9.1875, 11.6875,  8.4375,  ...,  0.0835,  0.0835,  0.0835],
         [11.6875, 13.1875,  8.8125,  ..., -0.0449, -0.0449, -0.0449],
         [12.1250, 13.1250,  8.3750,  ..., -0.3125, -0.3125, -0.3125],
         ...,
         [ 5.1875, 10.0000,  1.1094,  ..., -2.4062, -2.4062, -2.4062],
         [ 5.1562,  9.9375,  1.0781,  ..., -2.4219, -2.4219, -2.4219],
         [ 5.1250,  9.9375,  1.2109,  ..., -2.3750, -2.3750, -2.3750]]],
       device='cuda:0', dtype=torch.bfloat16)

#### generate

In [ ]:
test_input_data["input_ids"].shape

torch.Size([1, 2048])

In [14]:
result = model.generate(
    # input_ids=test_input_data["input_ids"],
    input_ids=torch.randint(
        low=0,
        high=128,
        # size=(1, 2),
        size=(32, 77),
        dtype=torch.long,
        device="cuda",
    ),
    # attention_mask=test_input_data["attention_mask"],
    max_new_tokens=20,
)
# tokenizer.
result

tensor([[    91,     77,     91,     57,     91,     89,     91,  53498,     61,
             71,     91,     64,     91,     82,     91,     16,     91,     17,
             91,     18],
        [    93,     67,     93,     61,     93,     79,     93,     79,     93,
             67,     93,     61,     93,     84,     93,     59,     93,     93,
             70,     93],
        [    82,      5,     63,     14,     67,      5,     14,     67,     63,
             14,     67,      5,     14,     67,     63,     14,     67,     63,
             14,     67],
        [    45,     59,     61,     77,     59,     63,     62,     82,     13,
             93,     59,     63,     38,     13,     93,      0,     38,     13,
             93,      0],
        [    71,      2,     86,     51,     59,     91,     62,     73,     32,
             59,     91,     62,     73,     32,     59,     91,     62,     73,
             32,     59],
        [    91,     32,     15,     59,     61,     41,    

In [ ]:
getattr(model, "memory_cell", None) is None

False

In [ ]:
getattr(model, "memory_cell", None).memory.shape[0]

16

### Optimize train code

In [ ]:
config = {
    "k2": -1,
    "max_n_segments": 128,
    "return_all_logits": False,
    "segment_size": 1024,
    "vary_n_segments": False,
}

In [1]:
from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

# model_name = "unsloth/Llama-3.2-1B-Instruct"
model_name = "alpindale/Llama-3.2-1B-Instruct"
config = AutoConfig.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
model = AutoModelForCausalLM.from_config(
    config
    # model_name,
    # torch_dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
    # device_map={"": 0},
)
# model = RMTForReasoning.from_pretrained(
#     model_name,
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
# )
cell = MemoryCell(
    model,
    num_mem_tokens=16,
)
model = RecurrentWrapper(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [1]:
# import os

# os.environ["https_proxy"] = "127.0.0.1:2334"
from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

# model_name = "alpindale/Llama-3.2-1B-Instruct"
# model_name = "gpt2"
# model_name = "openai-community/gpt2"
model_name = "model_checkpoints/gpt2"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    # device_map={"": 0},
)
model.cuda()
config = AutoConfig.from_pretrained(model_name)
# model = RMTForReasoning.from_pretrained(
#     model_name,
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
# )
cell = MemoryCell(
    model,
    # num_mem_tokens=16,
    num_mem_tokens=16,
)
model = RecurrentWrapper(
    cell,
    # segment_size=768,
    # segment_size=1024,
    segment_size=512,
    # max_n_segments=2,
    max_n_segments=32,
    # vary_n_segments=False,
    # vary_n_segments=True,
    k2=-1,
)
# model.load_state_dict(torch.load("/code/model_best.pt"))
tokenizer = AutoTokenizer.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
# model.load_state_dict(torch.load("./model_best.pt"))
model.load_state_dict(torch.load("pytorch_model.bin"))

<All keys matched successfully>

In [3]:
tokenizer("the capital of is", return_tensors="pt").to("cuda")

{'input_ids': tensor([[1169, 3139,  286,  318]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1]], device='cuda:0')}

In [26]:
from accelerate import Accelerator

accelerator = Accelerator()
model = accelerator.prepare(model)

In [3]:
from torch import tensor

with torch.no_grad():
    model_inputs = tokenizer(
        #     # "the capital of russia is Moscow. what is the capital of Moscow?",
        # "hello world",
        "Daniel went to the bedroom. Sandra went to the kitchen. Daniel moved to the kitchen. John went back to the bedroom. Daniel journeyed to the bedroom. Daniel travelled to the kitchen. Sandra went back to the bathroom. Sandra went to the kitchen. Where is Sandra?",
        # "Mary journeyed to the office. Mary moved to the hallway. Where is Mary?",
        # "Mary went to the kitchen. John went back to the kitchen. Daniel went back to the hallway. Daniel went to the bathroom. Sandra travelled to the bathroom. Sandra travelled to the bedroom. Daniel went to the kitchen. Daniel moved to the office. Where is Daniel?",
        # '<context>\n"Simon forced him to eat to excess, and to drink large quantities\nof wine, which he detested. He grew extremely fat without\nincreasing in height or strength." His aunt and sister, deprived of the\npleasure of tending him, had the pain of hearing his childish voice raised\nin the abominable songs his gaolers taught him. Mary went to the kitchen. The brutality of Simon\n"depraved at once the body and soul of his pupil. He called him the young\nwolf of the Temple. He treated him as the young of wild animals are\ntreated when taken from the mother and reduced to captivity,--at once\nintimidated by blows and enervated by taming. He punished for\nsensibility; he rewarded meanness; he encouraged vice; he made the child\nwait on him at table, sometimes striking him on the face with a knotted\ntowel, sometimes raising the poker and threatening to strike him with it." [Simon left the Temple to become a municipal officer. He was involved in\nthe overthrow of Robespierre, and guillotined the day after him, 29th\nJuly, 1794.] Yet when Simon was removed the poor young Prince\'s condition became even\nworse. His horrible loneliness induced an apathetic stupor to which any\nsuffering would have been preferable. "He passed his days without any\nkind of occupation; they did not allow him light in the evening. His\nkeepers never approached him but to give him food;" and on the rare\noccasions when they took him to the platform of the Tower, he was unable\nor unwilling to move about. When, in November, 1794, a commissary named\nGomin arrived at the Temple, disposed to treat the little prisoner with\nkindness, it was too late. "He took extreme care of my brother," says\nMadame Royale. "For a long time the unhappy child had been shut up in\ndarkness, and he was dying of fright. John went back to the kitchen. In free\nEngland, as in despotic Turkey, the privileges and obligations which the\nlaw tolerates or imposes, and all the benefits which their existence\nconfers on the community, are the creatures and conditions of a supreme\nauthority from which there is no appeal, whether the instrument by which\nthis authority makes its will known be an act of parliament or a ukase. This conception of temporal sovereignty, especially familiarised to our\ngeneration by the teaching of Austin, was carried by De Maistre into\ndiscussions upon the limits of the Papal power with great ingenuity and\nforce, and, if we accept the premisses, with great success. It should be said here, that throughout his book on the Pope, De Maistre\ntalks of Christianity exclusively as a statesman or a publicist would\ntalk about it; not theologically nor spiritually, but politically and\nsocially. The question with which he concerns himself is the utilisation\nof Christianity as a force to shape and organise a system of civilised\nsocieties; a study of the conditions under which this utilisation had\ntaken place in the earlier centuries of the era; and a deduction from\nthem of the conditions under which we might ensure a repetition of the\nprocess in changed modern circumstance. In the eighteenth century men\n</context>\n\nQuestion: Where is John?',
        # 'Simon forced him to eat to excess, and to drink large quantities\nof wine, which he detested. He grew extremely fat without\nincreasing in height or strength." His aunt and sister, deprived of the\npleasure of tending him, had the pain of hearing his childish voice raised\nin the abominable songs his gaolers taught him. Mary went to the kitchen. The brutality of Simon\n"depraved at once the body and soul of his pupil. He called him the young\nwolf of the Temple. He treated him as the young of wild animals are\ntreated when taken from the mother and reduced to captivity,--at once\nintimidated by blows and enervated by taming. He punished for\nsensibility; he rewarded meanness; he encouraged vice; he made the child\nwait on him at table, sometimes striking him on the face with a knotted\ntowel, sometimes raising the poker and threatening to strike him with it." [Simon left the Temple to become a municipal officer. He was involved in\nthe overthrow of Robespierre, and guillotined the day after him, 29th\nJuly, 1794.] Yet when Simon was removed the poor young Prince\'s condition became even\nworse. His horrible loneliness induced an apathetic stupor to which any\nsuffering would have been preferable. "He passed his days without any\nkind of occupation; they did not allow him light in the evening. His\nkeepers never approached him but to give him food;" and on the rare\noccasions when they took him to the platform of the Tower, he was unable\nor unwilling to move about. When, in November, 1794, a commissary named\nGomin arrived at the Temple, disposed to treat the little prisoner with\nkindness, it was too late. "He took extreme care of my brother," says\nMadame Royale. "For a long time the unhappy child had been shut up in\ndarkness, and he was dying of fright. John went back to the kitchen. In free\nEngland, as in despotic Turkey, the privileges and obligations which the\nlaw tolerates or imposes, and all the benefits which their existence\nconfers on the community, are the creatures and conditions of a supreme\nauthority from which there is no appeal, whether the instrument by which\nthis authority makes its will known be an act of parliament or a ukase. This conception of temporal sovereignty, especially familiarised to our\ngeneration by the teaching of Austin, was carried by De Maistre into\ndiscussions upon the limits of the Papal power with great ingenuity and\nforce, and, if we accept the premisses, with great success. It should be said here, that throughout his book on the Pope, De Maistre\ntalks of Christianity exclusively as a statesman or a publicist would\ntalk about it; not theologically nor spiritually, but politically and\nsocially. The question with which he concerns himself is the utilisation\nof Christianity as a force to shape and organise a system of civilised\nsocieties; a study of the conditions under which this utilisation had\ntaken place in the earlier centuries of the era; and a deduction from\nthem of the conditions under which we might ensure a repetition of the\nprocess in changed modern circumstance. In the eighteenth century men. Where is John?',
        return_tensors="pt",
        # add_special_tokens=False,
    ).to("cuda")
    result = model.generate(
        # input_ids=test_input_data["input_ids"],
        # input_ids=tokenizer(
        **model_inputs,  # ["input_ids"],
        max_new_tokens=20,
        # do_sample=False,
    )
output_ids = result[0][len(model_inputs.input_ids[0]) :].tolist()
tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'kitchen'

In [4]:
tokenizer.decode(result[0])

'<|begin_of_text|>Mary went to the kitchen. John went back to the kitchen. Daniel went back to the hallway. Daniel went to the bathroom. Sandra travelled to the bathroom. Sandra travelled to the bedroom. Daniel went to the kitchen. Daniel moved to the office. Where is Daniel?<|end_of_text|>'

''

In [18]:
tokenizer(
    # "the capital of russia is Moscow.\n\nwhat is the capital?",
    "Mary went to the kitchen. John went back to the kitchen. Daniel went back to the hallway. Daniel went to the bathroom. Sandra travelled to the bathroom. Sandra travelled to the bedroom. Daniel went to the kitchen. Daniel moved to the office. Where is Daniel?",
    return_tensors="pt",
    add_special_tokens=False,
)

{'input_ids': tensor([[42584,  4024,   311,   279,  9979,    13,  3842,  4024,  1203,   311,
           279,  9979,    13, 15469,  4024,  1203,   311,   279, 51902,    13,
         15469,  4024,   311,   279, 15197,    13, 56786, 46368,   311,   279,
         15197,    13, 56786, 46368,   311,   279, 14150,    13, 15469,  4024,
           311,   279,  9979,    13, 15469,  7882,   311,   279,  5274,    13,
         11208,   374, 15469,    30]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}

In [150]:
from datasets import load_dataset

task = "qa1"
seq_len = "0k"
ds = load_dataset("RMT-team/babilong", seq_len)[task]

In [214]:
model.cuda()

sample_num = 14
sample = ds[sample_num]
sample

input_text = f"{sample['input']} {sample['question']}"
model_inputs = tokenizer.encode_plus(
    input_text,
    return_tensors="pt",
).to("cuda")
with torch.no_grad():
    result = model.generate(**model_inputs, max_new_tokens=20)
output_ids = result[0][len(model_inputs.input_ids[0]) :].tolist()
prediction = tokenizer.decode(output_ids, skip_special_tokens=True).strip(
    "\n"
)  # .split()[0]

print(
    # f"Task: {input_text[:200]}\nPrediction: {prediction}\nAnswer: {sample['target']}\n{'Correct!' if sample['target'] == prediction.strip() else 'Wrong'}"
    f"Task: {input_text}\nPrediction: {prediction}\nAnswer: {sample['target']}\n{'Correct!' if sample['target'] == prediction.strip() else 'Wrong'}"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Task: Sandra moved to the kitchen. Sandra went back to the bathroom. Mary journeyed to the bedroom. John journeyed to the bathroom. Sandra moved to the bedroom. Sandra travelled to the hallway. Sandra travelled to the garden. Sandra travelled to the bathroom. Daniel journeyed to the kitchen. John journeyed to the office. Where is Sandra? 
Prediction: bathroom
Answer: bathroom
Correct!


In [192]:
output_ids

[]

In [155]:
prediction

''

In [13]:
model_input

{'input_ids': tensor([[128000,  42584,   4024,    311,    279,   9979,     13,   3842,   4024,
           1203,    311,    279,   9979,     13,  15469,   4024,   1203,    311,
            279,  51902,     13,  15469,   4024,    311,    279,  15197,     13,
          56786,  46368,    311,    279,  15197,     13,  56786,  46368,    311,
            279,  14150,     13,  15469,   4024,    311,    279,   9979,     13,
          15469,   7882,    311,    279,   5274,     13,  11208,    374,  15469,
             30,    220]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [11]:
prediction

' bathroom'